# MCP Authentication and OAuth Security

Build and attack-test a signed OAuth access-token boundary around an HTTP MCP server. The lab uses ephemeral keys and an in-process HTTP transport: no credential, identity provider, or external network is required.

## Objectives and safety boundary

We will verify signature, type, key, issuer, audience, time, scope, tenant, and revocation; observe 401/403 behavior; then separate transport authentication from tool authorization. Ephemeral test keys, an in-memory revocation set, and ASGI transport are learning fixtures—not production identity infrastructure.

In [ ]:
import runpy
from pathlib import Path

module = runpy.run_path(Path('lab.py'))
RESOURCE = module['RESOURCE']
ISSUER = module['ISSUER']
RESOURCE, ISSUER

## Four roles, two decisions

The resource owner grants access; the MCP client obtains and presents a token; the authorization server issues it; the MCP server validates it as an OAuth resource server. HTTP middleware authenticates the request. The tool then authorizes the validated subject/client/tenant for one action and resource.

In [ ]:
environment = module['build_environment']()
token = environment.authorization_server.issue()
access = await environment.verifier.verify_token(token)
assert access is not None
{
    'subject': access.subject,
    'client_id': access.client_id,
    'resource': access.resource,
    'scopes': access.scopes,
    'tenant': access.claims['tenant_id'],
}

## A signature is necessary, not sufficient

The verifier pins RS256 and `at+jwt`, chooses a known issuer key, verifies signature plus exact issuer/audience, enforces `exp`/`nbf`/`iat` and maximum lifetime, validates application claims, and checks revocation. A genuine token for another API is still invalid here.

In [ ]:
attacks = {
    'wrong issuer': {'issuer': 'https://attacker.example/'},
    'wrong audience': {'audience': 'https://inventory.example/api'},
    'not yet valid': {'not_before_offset': 30},
    'ID-token confusion': {'token_type': 'JWT'},
    'excessive lifetime': {'lifetime_seconds': 600},
}
reasons = {}
for name, kwargs in attacks.items():
    candidate = environment.authorization_server.issue(token_id=name, **kwargs)
    assert await environment.verifier.verify_token(candidate) is None
    reasons[name] = environment.verifier.events[-1].reason
reasons

## Exercise the actual HTTP boundary

The next requests traverse the SDK's bearer middleware. Missing or invalid credentials return 401 before protocol/tool work. A cryptographically valid token without the global `mcp:access` scope returns 403. Tokens in query strings are deliberately ignored.

In [ ]:
missing = await module['request_boundary'](environment, token=None)
wrong_audience = environment.authorization_server.issue(
    audience='https://inventory.example/api', token_id='http-wrong-aud'
)
wrong = await module['request_boundary'](environment, token=wrong_audience)
no_global_scope = environment.authorization_server.issue(
    scopes=(module['READ_SCOPE'],), token_id='no-global'
)
insufficient = await module['request_boundary'](environment, token=no_global_scope)
assert (missing.status_code, wrong.status_code, insufficient.status_code) == (401, 401, 403)
{
    'missing': missing.json(),
    'challenge': missing.headers['www-authenticate'],
    'wrong_audience': wrong.json(),
    'insufficient_scope': insufficient.json(),
}

In [ ]:
metadata = await module['protected_resource_metadata'](environment)
assert metadata['resource'] == RESOURCE
assert metadata['authorization_servers'] == [ISSUER]
metadata

## Normal → attack → defense → retest

The complete scenario starts a real Streamable HTTP app, performs an allowed ticket call, denies a cross-tenant record with the same external shape as a missing record, revokes the token, and confirms the raw bearer value is absent from evidence.

In [ ]:
evidence = await module['run_scenario']()
assert evidence['valid_call_is_error'] is False
assert evidence['cross_tenant_is_error'] is True
assert evidence['revoked_status'] == 401
assert evidence['raw_token_in_evidence'] is False
evidence

## Rotation, revocation, and replay

During rotation, publish a new key before using it and retain the old verification key until bounded tokens expire. Revocation can deny a `jti`, but a `jti` does not make a bearer token one-time: possession permits replay until expiry or effective revocation. Sender-constrained tokens can reduce this risk where supported.

In [ ]:
rotation = module['build_environment']()
old = rotation.authorization_server.issue(token_id='old')
rotation.authorization_server.rotate('key-2')
new = rotation.authorization_server.issue(token_id='new')
assert await rotation.verifier.verify_token(old) is not None
assert await rotation.verifier.verify_token(new) is not None
rotation.authorization_server.retire('key-1')
assert await rotation.verifier.verify_token(old) is None
rotation.verifier.events[-1]

## Practical extensions

1. Add a `ticket:export` scope boundary and assert a 403 scope challenge.
2. Replace direct key lookup with a bounded, issuer-pinned JWKS cache; test unknown-key refresh and issuer outage.
3. Implement a fake RFC 7662 introspection verifier and compare centralized revocation with JWT availability.
4. Add a second downstream audience and prove the inbound MCP token is never forwarded.
5. Run the focused suite: `python -m pytest -q tests/test_course_06_oauth_security.py`.

## Reflection

Why is a valid `mcp:access` token still insufficient to read `globex-200`? Which component should obtain a different credential if `ticket.read` must call a downstream ticket API? What deployment evidence would you still need before calling this production-ready?